# Goal 4 -- Autonomous Compliance Sentinel Agent
### SRH Heidelberg | Data Ethics and Responsible AI
**Authors:** Vikrant Singh and Kay Marc Muller

---

## What is this notebook?

This notebook builds the **Goal 4 Agent**, the final piece of the Autonomous Compliance Sentinel project.

The system reads internal project proposals (fetched from Jira), checks whether they violate any of our 9 Responsible AI policies (RAI-01 through RAI-09), and writes the verdict back to Jira as a comment.

---

## The full pipeline

```
Jira Project (DE2)
        |
        v
fetch_proposals() -- pulls all issues and their descriptions
        |
        v
For each proposal:
        |
        v
[Signal 1] TF-IDF + Logistic Regression Classifier
        |
        | -- Compliant? --> post Compliant comment to Jira
        |
        | -- Red Flag? --> Signal 2
        v
[Signal 2] LLM Agent + 6 Tools
        |
        v
Human-in-the-Loop Gate (High severity = human must approve)
        |
        v
Post verdict as Jira comment + save results to JSON
```

---

## Two signals, not one

| Signal | What it is | When it runs |
|--------|-----------|--------------|
| Signal 1 | TF-IDF + Logistic Regression, threshold=0.4 | Always, on every proposal |
| Signal 2 | LLM + 6 tools | Only when Signal 1 flags Red Flag |

---

## The 9 RAI Policies

| Policy | Name | Severity |
|--------|------|----------|
| RAI-01 | Data Protection | High |
| RAI-02 | Transparency | High |
| RAI-03 | Fairness | High |
| RAI-04 | Human Dignity and Vulnerable Groups | High |
| RAI-05 | Prohibited Purpose | High |
| RAI-06 | Security | Medium |
| RAI-07 | Human Oversight | Medium |
| RAI-08 | Data Minimisation | Medium |
| RAI-09 | Explainability | Medium |

## The 6 Tools

| Tool | Role |
|------|------|
| `search_policies` | RAG: find relevant policy chunks |
| `get_trigger_words` | XAI: which words drove the classifier |
| `get_policy_severity` | Fixed severity lookup |
| `log_decision` | Article 12 audit trail |
| `write_compliance_verdict` | Structured final verdict |
| `llm_assessment` | Free-form LLM reasoning about the proposal |

---
## Step 0 -- Install dependencies

Run this cell once when setting up a new environment.

In [1]:
%pip install langchain langgraph langchain-community langchain-chroma langchain-huggingface langchain-groq sentence-transformers pdfplumber requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


---
## Step 1 -- Imports and environment variables

We load the `.env` file first, before anything else, so every cell below has access to all credentials without needing to reload.

**Group 1 -- Environment and credentials**
All secrets (Groq API key, Jira token, Jira email, Jira server, Jira project key) are loaded from `.env`.
Nothing is hardcoded in this notebook.

**Group 2 -- Signal 1 (the ML classifier)**
`joblib` reloads the trained pipeline. `sklearn` and `spacy` pieces must be imported so `joblib` can find the class definitions (`ThresholdAdjustor`, `token_pos`) when unpickling.

**Group 3 -- PDF loading and chunking**
`PDFPlumberLoader` reads the policy catalogue. `RecursiveCharacterTextSplitter` cuts pages into smaller chunks.

**Group 4 -- Embedding and vector storage**
`HuggingFaceEmbeddings` converts text to vectors. `Chroma` stores and searches those vectors.

**Group 5 -- Tools and LLM**
`tool` decorator turns Python functions into LLM-callable tools. `ChatGroq` connects to llama-3.1-8b-instant.

**Group 6 -- Agent (LangGraph)**
`StateGraph`, `START`, `END` build the graph. `ToolNode` runs tool calls. `MemorySaver` saves state so the human gate can pause.

**Group 7 -- Jira integration**
`requests` makes HTTP calls to the Jira REST API v3. `HTTPBasicAuth` handles authentication.

In [2]:
# Group 1 -- Environment: load first, before everything else
from dotenv import load_dotenv
import os
load_dotenv(r"C:\MyGithubSpace\Data-Ethics\.env", override=True)

# Group 2 -- Signal 1: reloading the already-trained classifier
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import spacy

# Group 3 -- PDF loading and chunking
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Group 4 -- Embedding and vector storage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Group 5 -- Tools and LLM
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Group 6 -- Agent (LangGraph)
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

# Group 7 -- Jira integration
import requests
from requests.auth import HTTPBasicAuth
import json
import datetime

print("All imports OK")
print("GROQ key loaded:", bool(os.environ.get("GROQ_API_KEY")))
print("Jira server loaded:", bool(os.environ.get("JIRA_SERVER")))

C:\Users\Vikrant Singh Thakur\AppData\Local\Temp\ipykernel_19868\495181328.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PDFPlumberLoader


All imports OK
GROQ key loaded: True
Jira server loaded: True


---
## Step 2a -- Reload Signal 1 (the trained classifier)

The classifier was trained in Goal 2 and saved as `logRegpipelineV2.pkl`. We reload it here rather than retrain it. Same weights = same predictions = reproducible results.

`ThresholdAdjustor`, `token_pos`, `nlp`, `explain_prediction`, `audit_trail` must be imported from `xai2.py` BEFORE calling `joblib.load()`. Without them, joblib throws `AttributeError` because it cannot find the class definitions.

In [3]:
from xai2 import ThresholdAdjustor, token_pos, nlp, explain_prediction, audit_trail

classifier_pipeline = joblib.load("logRegpipelineV2.pkl")
print("Signal 1 loaded.")
print("Pipeline steps:", list(classifier_pipeline.named_steps.keys()))

Signal 1 loaded.
Pipeline steps: ['tfidf', 'clf']


---
## Step 2b -- Load the RAI Policy Catalogue PDF

`PDFPlumberLoader` reads the catalogue page by page and returns one `Document` object per page. Each `Document` has `.page_content` (the text) and `.metadata` (page number, source path).

In [4]:
path = "../../data/RAI_Policy_Catalogue.pdf"

loader = PDFPlumberLoader(path)
raw_pages = loader.load()

print(f"Pages loaded: {len(raw_pages)}")
print("First 300 characters of page 1:")
print(raw_pages[0].page_content[:300])

Pages loaded: 10
First 300 characters of page 1:
Autonomous Compliance Sentinel
Responsible AI Policy Catalogue, RAI-01 through RAI-09
SRH Heidelberg, Data Ethics and Responsible AI module. Author: Vikrant Singh and Kay Marc
Muller. This catalogue is the retrieval corpus for the Goal 4 compliance agent. Each policy below is
written in the same fix


---
## Step 3 -- Chunking

`RecursiveCharacterTextSplitter` cuts each page into smaller, more targeted pieces for vector search. It tries separators in priority order: paragraph breaks, newlines, sentence endings, spaces, mid-word.

**Settings:** `chunk_size=600` (roughly 4-6 sentences), `chunk_overlap=80` (prevents mid-sentence cuts at boundaries).

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_pages)
print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk text:")
print(chunks[0].page_content)

Total chunks created: 33

First chunk text:
Autonomous Compliance Sentinel
Responsible AI Policy Catalogue, RAI-01 through RAI-09
SRH Heidelberg, Data Ethics and Responsible AI module. Author: Vikrant Singh and Kay Marc
Muller. This catalogue is the retrieval corpus for the Goal 4 compliance agent. Each policy below is
written in the same fixed order every time, Policy ID and name, Severity, Regulatory Basis,
Requirement, Violation Patterns, and a Compliant Example, so that the meaning of any retrieved
chunk stays clear even without the rest of the document around it. Severity is a fixed lookup value,


---
## Step 4 -- Embedding

`BAAI/bge-small-en-v1.5` converts each text chunk into a 384-dimensional vector. Two chunks with similar meaning produce numerically similar vectors even if they use different words. `normalize_embeddings=True` ensures cosine similarity measures angle (meaning) not magnitude.

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.


---
## Step 5 -- Vector Database (Chroma)

Chroma stores embedded chunks on disk and provides fast cosine similarity search. `collection_name="rai_policies_v2"` keeps this separate from earlier collections.

**IMPORTANT:** `Chroma.from_documents()` APPENDS to existing collections. If you rerun this cell, call `vector_db.delete_collection()` first to avoid duplicates.

In [7]:
# If rebuilding: uncomment the two lines below first
# vector_db = Chroma(collection_name="rai_policies_v2", embedding_function=embeddings, persist_directory="Chroma")
# vector_db.delete_collection()

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rai_policies_v2",
    persist_directory="Chroma"
)

count = vector_db._collection.count()
print(f"Chunks stored in Chroma: {count}")
print(f"Match len(chunks): {count == len(chunks)}")

Chunks stored in Chroma: 495
Match len(chunks): False


---
## Step 6 -- Tools

A tool is a Python function the LLM can call by name. The `@tool` decorator turns a plain function into a `Tool` object. The LLM reads the docstring to decide WHEN to call it, and the type hints to know WHAT arguments to send.

**All 6 tools:**

| Tool | Role |
|------|------|
| `search_policies` | RAG: vector search in Chroma, returns policy chunks |
| `get_trigger_words` | XAI: contribution = tfidf_score x weight, only present words |
| `get_policy_severity` | Fixed lookup: RAI-01 to 05 = High, RAI-06 to 09 = Medium |
| `log_decision` | Article 12 audit trail |
| `write_compliance_verdict` | Structured final output |
| `llm_assessment` | Free-form LLM reasoning about the proposal |

### Tool 1 -- search_policies (RAG Retrieval)

Searches the Chroma vector database for policy chunks relevant to a query. The LLM calls this with a query based on trigger words it already found. Deduplication via a Python `set` prevents the same chunk appearing multiple times.

In [8]:
@tool
def search_policies(query:str, k:int=3)->str:
    """
    Search the RAI policy catalogue for clauses relevant to a query.
    Use this to find which of the 9 RAI policies apply to a proposal.
    Args:
        query: a short description of the concern or topic to search for
        k: number of matching policy chunks to return
    Returns:
        the matched policy text chunks, each labelled with its source page
    """
    results = vector_db.similarity_search(query, k=k)
    output = ""
    seen = set()
    for doc in results:
        page = doc.metadata.get("page", "unknown")
        fingerprint = doc.page_content[:100].strip()
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        output += f"[page {page}]\n{doc.page_content}\n\n"
    return output

### Tool 2 -- get_trigger_words (XAI Grounding)

Computes per-proposal word contributions: `contribution = tfidf_score x weight`. Only words actually present in the proposal have a non-zero tfidf_score. Grounds the LLM's reasoning in real classifier evidence. Satisfies EU AI Act Article 14.

In [9]:
@tool
def get_trigger_words(text:str, n:int=10)->str:
    """
    Find which words in this specific proposal drove the classifier's prediction.
    Use this to ground your reasoning in the model's actual evidence for this proposal.
    Args:
        text: the proposal text to explain
        n: how many top contributing words to return
    Returns:
        the top words with their contribution scores and direction
    """
    results = explain_prediction(classifier_pipeline, text, n)
    output = ""
    for word, score in results:
        direction = "toward Red Flag" if score > 0 else "toward Compliant"
        output += f"{word}: {round(score, 4)} ({direction})\n"
    return output

### Tool 3 -- get_policy_severity (Fixed Lookup)

Severity is NEVER decided by the LLM. It is a fixed value from the catalogue. A deterministic lookup table guarantees the same answer every time, required for Article 12 reproducibility.

In [10]:
@tool
def get_policy_severity(policy_id:str)->str:
    """
    Look up the fixed severity level for a given RAI policy.
    Severity is a fixed value from the policy catalogue, not something to reason about.
    Use this to determine whether a violation needs human approval (High) or can proceed automatically (Medium).
    Args:
        policy_id: the policy identifier, e.g. RAI-01, RAI-02, up to RAI-09
    Returns:
        the severity level, either High or Medium, and what it means for routing
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), None)
    if severity is None:
        return f"{policy_id} is not a recognised RAI policy. Valid options are RAI-01 through RAI-09."
    if severity == "High":
        return f"{policy_id} is High severity. This violation must be routed to a human reviewer for approval before any action is taken."
    return f"{policy_id} is Medium severity. This violation may proceed to automatic remediation with an optional spot check." 

### Tool 4 -- log_decision (EU AI Act Article 12 Audit Trail)

Every compliance decision must be logged with enough information for an auditor to reconstruct it later. `explanation_method` is hardcoded as 'LogReg coef_ x tfidf' because LogReg coefficients are exact and deterministic. Same input always produces same log entry.

In [11]:
@tool
def log_decision(text:str, y_true:int, y_pred:int, proba:float, n_terms:int=10)->str:
    """
    Log a compliance decision for audit purposes.
    Use this once you have reached a conclusion about a proposal.
    Satisfies EU AI Act Article 12: every decision is logged with the exact inputs,
    prediction, probability, and trigger words that produced it.
    Args:
        text: the proposal text that was assessed
        y_true: the true label, 1 for Red Flag, 0 for Compliant
        y_pred: the predicted label, 1 for Red Flag, 0 for Compliant
        proba: the classifier's predicted probability of Red Flag
        n_terms: how many trigger words to include in the log
    Returns:
        a formatted string of the full audit log entry
    """
    log = audit_trail(classifier_pipeline, text, y_true, y_pred, proba, n_terms)
    output = ""
    output += f"Actual label      : {'Red Flag' if log['actual']==1 else 'Compliant'}\n"
    output += f"Predicted label   : {'Red Flag' if log['predicted']==1 else 'Compliant'}\n"
    output += f"Probability       : {log['probability']}\n"
    output += f"Correct           : {log['correct']}\n"
    output += f"Trigger words     : {', '.join(log['trigger_words'])}\n"
    output += f"Explanation method: {log['explanation_method']}\n"
    return output

### Tool 5 -- write_compliance_verdict (Structured Final Output)

The last tool the LLM calls. `policy_id` comes from `search_policies` results. `reason` is written by the LLM grounded in trigger words. `recommended_fix` is written by the LLM grounded in retrieved policy text. Severity is looked up deterministically inside the tool.

In [12]:
@tool
def write_compliance_verdict(policy_id:str, reason:str, recommended_fix:str)->str:
    """
    Write the final compliance verdict for a flagged proposal.
    Call this as your last action after log_decision.
    Args:
        policy_id: the policy that was violated, e.g. RAI-02
        reason: one sentence explaining what the proposal is missing, grounded in the trigger words
        recommended_fix: one sentence describing what the proposal must add to become compliant
    Returns:
        a formatted compliance verdict string
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), "Unknown")
    output = ""
    output += f"POLICY VIOLATED  : {policy_id.upper()}\n"
    output += f"SEVERITY         : {severity}\n"
    output += f"REASON           : {reason}\n"
    output += f"RECOMMENDED FIX  : {recommended_fix}\n"
    return output

### Tool 6 -- llm_assessment (Free-form LLM Reasoning)

This is the new tool. Unlike the other 5 tools which produce structured, deterministic outputs, this tool asks the LLM to reason freely about the proposal in plain language.

It gives the LLM a chance to say things like: "This proposal is borderline because...", "The concern here is not just one policy but a combination of...", or "While the classifier flagged this, the actual risk level seems low because..."

This is the human-readable narrative layer on top of the structured verdict. Your friend can display this on the Streamlit dashboard as the "AI commentary" section.

In [13]:
@tool
def llm_assessment(proposal_text:str)->str:
    """
    Generate a free-form LLM assessment of the proposal in plain language.
    Call this after you have gathered evidence from other tools.
    Use this to provide a human-readable narrative explanation of your findings,
    including any nuances, borderline concerns, or context that the structured
    verdict cannot capture.
    Args:
        proposal_text: the full proposal text to assess
    Returns:
        a plain language narrative assessment of the proposal
    """
    from langchain_groq import ChatGroq
    from langchain_core.messages import SystemMessage, HumanMessage

    assessment_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.3,
        api_key=os.environ.get("GROQ_API_KEY")
    )

    system = SystemMessage(content="""You are a Responsible AI compliance expert reviewing internal project proposals.
Write a clear, plain-language assessment of this proposal for a human reviewer.
Cover: what the proposal does, what the main compliance concern is, how serious it is, and what the team should focus on fixing first.
Write in 3-5 sentences. Be direct and practical.""")

    human = HumanMessage(content=f"Proposal to assess:\n\n{proposal_text}")

    response = assessment_llm.invoke([system, human])
    return response.content

---
## Step 7 -- LLM Setup

`llama-3.1-8b-instant` via Groq free API. `temperature=0.0` means fully deterministic for the agent (required for Article 12). Note: Tool 6 (`llm_assessment`) uses `temperature=0.3` separately to allow more natural language in its narrative response.

`bind_tools(tools)` attaches all 6 tools to the LLM permanently.

In [14]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    api_key=os.environ.get("GROQ_API_KEY")
)

tools = [
    search_policies,
    get_trigger_words,
    get_policy_severity,
    log_decision,
    write_compliance_verdict,
    llm_assessment,
]

llm_with_tools = llm.bind_tools(tools)
print("LLM loaded and tools bound.")
print(f"Tools available to LLM: {[t.name for t in tools]}")

LLM loaded and tools bound.
Tools available to LLM: ['search_policies', 'get_trigger_words', 'get_policy_severity', 'log_decision', 'write_compliance_verdict', 'llm_assessment']


---
## Step 8 -- LangGraph Agent

### Step 8a -- AgentState (Shared Memory)

AgentState is a TypedDict -- a dictionary with fixed key names and declared types. Every node reads from it and writes back to it. `messages: Annotated[list, operator.add]` means new messages are APPENDED not overwritten, keeping the full conversation history across multiple tool calls.

In [15]:
class AgentState(TypedDict):
    # The raw proposal text being assessed. Set at the start, never changes.
    proposal_text: str
    # The classifier's hard prediction. 1 = Red Flag, 0 = Compliant.
    y_pred: int
    # The classifier's Red Flag probability score. e.g. 0.6427
    proba: float
    # High or Medium. Written by the LLM after calling get_policy_severity.
    severity: str
    # Full conversation history. Annotated means messages are APPENDED not overwritten.
    messages: Annotated[list, operator.add]
    # True if human approved or Medium severity auto-approved. False if rejected.
    human_approved: bool
    # Plain English summary of the final outcome.
    final_decision: str

print("AgentState defined.")

AgentState defined.


### Step 8b -- Nodes

In [16]:
def classify(state:AgentState)->dict:
    # Read the proposal text from shared state
    text = state["proposal_text"]
    # predict_proba returns shape (1, 2): [[prob_compliant, prob_red_flag]]
    # [0][1] gets the Red Flag probability for the first (only) document
    proba = classifier_pipeline.predict_proba([text])[0][1]
    # predict() applies the threshold (0.4) internally via ThresholdAdjustor
    # Returns 1 for Red Flag, 0 for Compliant
    # int() converts numpy.int64 to plain Python int
    y_pred = int(classifier_pipeline.predict([text])[0])
    # Return only the fields this node changes
    # LangGraph merges this partial dict into AgentState automatically
    return {
        "y_pred": y_pred,
        "proba": round(float(proba), 4),
    }

def agent(state:AgentState)->dict:
    system=SystemMessage(content="""You are a compliance assessment agent for a Responsible AI policy system.

You have access to these tools:
- {get_trigger_words}: find which words in the proposal drove the classifier prediction
- {search_policies}: search the RAI policy catalogue for relevant clauses
- {get_policy_severity}: look up the severity of a specific RAI policy
- {log_decision}: write the audit trail entry for this decision
- {write_compliance_verdict}: write the final structured compliance verdict
- {llm_assessment}: generate a free-form plain language assessment of the proposal

Use your tools in whatever order and however many times you judge necessary to accurately assess whether this proposal violates any RAI policy. Call {llm_assessment} to provide a narrative explanation. When you are confident in your assessment, call {log_decision} and then {write_compliance_verdict} to conclude.""")

    messages=[system]+state["messages"]
    response=llm_with_tools.invoke(messages)
    return{"messages":[response]}

def human_gate(state:AgentState)->dict:
    severity=state["severity"]

    if severity=="Medium":
        print("Medium severity: proceeding automatically.")
        return{
            "human_approved":True,
            "final_decision":"Medium severity violation. Automatic remediation approved."
        }

    print("\n--- HIGH SEVERITY: HUMAN APPROVAL REQUIRED ---")
    print(f"Proposal text:\n{state['proposal_text']}\n")
    print("Agent reasoning and tool outputs:")
    for message in state["messages"]:
        if hasattr(message, "name") and message.name and message.content:
            print(f"[{message.name}] {message.content}")
        elif message.content and not hasattr(message, "tool_calls"):
            print(f"[LLM] {message.content}")
    print("----------------------------------------------")

    decision=input("Type APPROVE or REJECT: ").strip().upper()

    if decision=="APPROVE":
        return{
            "human_approved":True,
            "final_decision":"High severity violation. Human reviewer approved remediation."
        }

    return{
        "human_approved":False,
        "final_decision":"High severity violation. Human reviewer rejected remediation. No changes made."
    }

print("Nodes defined.")

Nodes defined.


### Step 8c -- Routing Functions

Routing functions are plain Python functions (not nodes) that LangGraph calls after a node finishes. They receive state and return the name of the next node, or END to stop.

In [17]:
def route_after_classify(state:AgentState)->str:
    if state["y_pred"] == 1:
        return "agent"
    return END

def route_after_agent(state:AgentState)->str:
    last_message = state["messages"][-1]
    # [-1]: the very last item in the messages list
    # hasattr: built-in that returns True if the object has that attribute
    # len(...) > 0: confirms the LLM actually requested a tool (not an empty list)
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return "tools"
    return "human_gate"

print("Routing functions defined.")

Routing functions defined.


### Step 8d -- Build and Compile the Graph

In [18]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("classify", classify)
graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_node("human_gate", human_gate)

graph_builder.add_edge(START, "classify")

graph_builder.add_conditional_edges(
    "classify",
    route_after_classify,
    {"agent": "agent", END: END}
)

graph_builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "human_gate": "human_gate"}
)

graph_builder.add_edge("tools", "agent")
graph_builder.add_edge("human_gate", END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("Graph compiled successfully.")

Graph compiled successfully.


---
## Step 8e -- Run the Agent on a Single Proposal

Use this cell to test the agent on one proposal before running the full batch.
Change `thread_id` every time you run a new proposal.

In [19]:
config = {"configurable": {"thread_id": "test_run_1"}}

initial_state = {
    "proposal_text": "Our chatbot will handle customer complaints automatically without telling users it is an AI.",
    "y_pred": 0,
    "proba": 0.0,
    "severity": "",
    "messages": [HumanMessage(content="Our chatbot will handle customer complaints automatically without telling users it is an AI.")],
    "human_approved": False,
    "final_decision": "",
}

result = graph.invoke(initial_state, config=config)

print("\n--- FINAL DECISION ---")
print(result["final_decision"])
print(f"Classifier prediction : {'Red Flag' if result['y_pred']==1 else 'Compliant'}")
print(f"Probability           : {result['proba']}")
print(f"Human approved        : {result['human_approved']}")

print("\n--- COMPLIANCE VERDICT ---")
for message in reversed(result["messages"]):
    if hasattr(message, "name") and message.name == "write_compliance_verdict":
        print(message.content)
        break

print("\n--- LLM NARRATIVE ASSESSMENT ---")
for message in reversed(result["messages"]):
    if hasattr(message, "name") and message.name == "llm_assessment":
        print(message.content)
        break


--- HIGH SEVERITY: HUMAN APPROVAL REQUIRED ---
Proposal text:
Our chatbot will handle customer complaints automatically without telling users it is an AI.

Agent reasoning and tool outputs:
[LLM] Our chatbot will handle customer complaints automatically without telling users it is an AI.
[search_policies] [page 2]
Violation Patterns
A proposal describes a chatbot, virtual assistant, or automated messaging system with no mention of
disclosing to the user that they are talking to an AI system. A proposal describes an automated
decision, such as an approval, denial, or score, being delivered to a customer with no statement of
how or whether the customer is told an algorithm was involved.
Compliant Example
A proposal for a support chatbot states that every conversation opens with a clear statement that the
user is speaking with an automated assistant, and that a human agent is one request away at any


[get_trigger_words] it: 0.4499 (toward Red Flag)
without: 0.1494 (toward Red Flag)
our:

---
## Step 9 -- Jira Integration

### Step 9a -- Connect to Jira

All credentials come from `.env`. Nothing is hardcoded.

- `JIRA_SERVER`: your Atlassian site URL
- `JIRA_EMAIL`: your Atlassian account email
- `JIRA_API_TOKEN`: your personal API token (starts with ATATT)
- `JIRA_PROJECT_KEY`: the project key (DE2)

In [20]:
JIRA_SERVER = os.environ.get("JIRA_SERVER")
JIRA_EMAIL = os.environ.get("JIRA_EMAIL")
JIRA_TOKEN = os.environ.get("JIRA_API_TOKEN")

jira_auth = HTTPBasicAuth(JIRA_EMAIL, JIRA_TOKEN)
jira_headers = {"Accept": "application/json", "Content-Type": "application/json"}

# Test the connection
response = requests.get(
    f"{JIRA_SERVER}/rest/api/3/myself",
    auth=jira_auth,
    headers=jira_headers
)
print("Status code:", response.status_code)
print("Connected as:", response.json()["displayName"])
print("Email:", response.json()["emailAddress"])

Status code: 200
Connected as: Prashant THAKUR
Email: thakurseema1304@gmail.com


### Step 9b -- Fetch Proposals from Jira

`/rest/api/3/search/jql` is the Jira REST API v3 search endpoint (the older `/rest/api/3/search` was deprecated in 2026).

Jira API v3 returns descriptions as Atlassian Document Format (nested JSON), not plain text. The inner loop walks through the nested content blocks and extracts just the text.

In [21]:
def fetch_proposals(project_key=None):
    if project_key is None:
        project_key = os.environ.get("JIRA_PROJECT_KEY")

    url = f"{JIRA_SERVER}/rest/api/3/search/jql"

    params = {
        "jql": f"project = {project_key} ORDER BY created ASC",
        "maxResults": 50,
        "fields": "summary,description"
    }

    response = requests.get(
        url,
        auth=jira_auth,
        headers=jira_headers,
        params=params
    )

    data = response.json()
    proposals = []
    for issue in data["issues"]:
        description_obj = issue["fields"].get("description")
        if description_obj is None:
            description_text = ""
        else:
            # Jira API v3 returns description as Atlassian Document Format (nested JSON)
            # We walk through the nested content blocks to extract plain text
            description_text = ""
            for block in description_obj.get("content", []):
                for inline in block.get("content", []):
                    if inline.get("type") == "text":
                        description_text += inline.get("text", "")
                description_text += "\n"

        proposals.append({
            "issue_key": issue["key"],
            "summary": issue["fields"]["summary"],
            "description": description_text.strip()
        })

    return proposals

proposals = fetch_proposals()
print(f"Fetched {len(proposals)} proposals from Jira.")
for p in proposals:
    print(f"{p['issue_key']}: {p['summary']}")
    print(f"  Preview: {p['description'][:80]}...")

Fetched 17 proposals from Jira.
DE2-1: Issue 1 -- RAI-02 Red Flag (Transparency)
  Preview: Our customer support chatbot will handle all incoming queries automatically.It w...
DE2-2: Issue 2 -- RAI-01 Red Flag (Data Protection)
  Preview: The loan processing system will collect applicant financial records, credithisto...
DE2-3: Issue 3 -- RAI-07 Red Flag (Human Oversight)
  Preview: The automated fraud detection system will apply model outputs directly tocustome...
DE2-4: Issue 4 -- RAI-03 Red Flag (Fairness)
  Preview: The resume screening tool will train on historical hiring decisions fromthe past...
DE2-5: Issue 5 -- RAI-05 Red Flag (Prohibited Purpose)
  Preview: The customer loyalty platform will assign each customer a persistent trustscore ...
DE2-6: Issue 6 -- RAI-09 Red Flag (Explainability)
  Preview: The content moderation model will automatically remove posts that violatecommuni...
DE2-7: Issue 7 -- Compliant (RAI-02)
  Preview: Our virtual assistant will handle tier-one cus

### Step 9c -- Post Compliance Verdict Back to Jira

`/rest/api/3/issue/{issue_key}/comment` posts a comment on a Jira issue.

Jira API v3 requires comments in Atlassian Document Format (ADF), not plain text. The `_build_adf_comment()` helper converts our verdict string into the correct nested JSON structure that Jira expects.

In [22]:
def _build_adf_comment(text:str)->dict:
    # Jira API v3 requires comments in Atlassian Document Format (ADF)
    # This converts plain text into the required nested JSON structure
    # Each line becomes a paragraph node containing a text node
    paragraphs = []
    for line in text.split("\n"):
        if line.strip():
            paragraphs.append({
                "type": "paragraph",
                "content": [{"type": "text", "text": line}]
            })

    return {
        "body": {
            "version": 1,
            "type": "doc",
            "content": paragraphs if paragraphs else [
                {"type": "paragraph", "content": [{"type": "text", "text": text}]}
            ]
        }
    }

def post_jira_comment(issue_key:str, comment_text:str)->bool:
    # Posts a comment on a Jira issue
    # Returns True if successful, False if not
    url = f"{JIRA_SERVER}/rest/api/3/issue/{issue_key}/comment"
    payload = _build_adf_comment(comment_text)

    response = requests.post(
        url,
        auth=jira_auth,
        headers=jira_headers,
        json=payload
    )

    if response.status_code == 201:
        print(f"  Comment posted to {issue_key} successfully.")
        return True
    else:
        print(f"  Failed to post comment to {issue_key}. Status: {response.status_code}")
        print(f"  Response: {response.text[:200]}")
        return False

print("Jira comment functions defined.")

Jira comment functions defined.


---
## Step 10 -- Batch Pipeline: Fetch, Assess, Write Back

### Step 10a -- Build batch graph (auto-approve)

For batch mode we swap the `human_gate` node for an auto-approving version. This prevents `input()` from blocking the batch run. High severity proposals are still flagged clearly in the Jira comment so your friend can build a Streamlit approve/reject UI on top of the results.

In [23]:
def build_batch_graph():
    # Build a fresh graph with auto-approve human gate
    # Everything else stays exactly the same as the main graph

    def auto_human_gate(state:AgentState)->dict:
        severity = state["severity"]
        if severity == "Medium":
            return {
                "human_approved": True,
                "final_decision": "Medium severity violation. Automatic remediation approved."
            }
        # High severity: auto-approve but flag clearly for human review on dashboard
        return {
            "human_approved": True,
            "final_decision": "High severity violation. Auto-approved in batch mode. PENDING HUMAN REVIEW on dashboard."
        }

    batch_builder = StateGraph(AgentState)
    batch_builder.add_node("classify", classify)
    batch_builder.add_node("agent", agent)
    batch_builder.add_node("tools", ToolNode(tools))
    batch_builder.add_node("human_gate", auto_human_gate)

    batch_builder.add_edge(START, "classify")
    batch_builder.add_conditional_edges(
        "classify",
        route_after_classify,
        {"agent": "agent", END: END}
    )
    batch_builder.add_conditional_edges(
        "agent",
        route_after_agent,
        {"tools": "tools", "human_gate": "human_gate"}
    )
    batch_builder.add_edge("tools", "agent")
    batch_builder.add_edge("human_gate", END)

    batch_memory = MemorySaver()
    return batch_builder.compile(checkpointer=batch_memory)

batch_graph = build_batch_graph()
print("Batch graph compiled.")

Batch graph compiled.


### Step 10b -- Run the full batch pipeline

In [24]:
def run_batch_pipeline(proposals):
    results = []
    batch_graph = build_batch_graph()

    for p in proposals:
        print(f"\nProcessing {p['issue_key']}: {p['summary']}")

        if not p["description"]:
            print(f"  Skipping -- no description.")
            continue

        # Signal 1
        proba = float(classifier_pipeline.predict_proba([p["description"]])[0][1])
        y_pred = int(classifier_pipeline.predict([p["description"]])[0])
        print(f"  Signal 1: {'Red Flag' if y_pred == 1 else 'Compliant'} (proba={round(proba,4)})")

        if y_pred == 0:
            # Compliant -- skip agent entirely
            entry = {
                "issue_key": p["issue_key"],
                "summary": p["summary"],
                "y_pred": 0,
                "proba": round(proba, 4),
                "prediction": "Compliant",
                "policy_id": None,
                "severity": None,
                "trigger_words": None,
                "reason": None,
                "recommended_fix": None,
                "llm_narrative": None,
                "final_decision": "Compliant -- no RAI violation detected.",
                "human_approved": True,
                "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            }
            results.append(entry)

            # Post compliant comment to Jira
            comment = (
                f"COMPLIANCE SENTINEL ASSESSMENT\n"
                f"{'='*40}\n"
                f"Issue       : {p['issue_key']}\n"
                f"Prediction  : Compliant\n"
                f"Probability : {round(proba, 4)}\n"
                f"Decision    : No RAI violation detected. No action required.\n"
                f"Assessed at : {entry['timestamp']}"
            )
            post_jira_comment(p["issue_key"], comment)
            continue

        # Signal 2 -- run agent with auto-approve
        config = {"configurable": {"thread_id": p["issue_key"] + "_batch"}}
        initial_state = {
            "proposal_text": p["description"],
            "y_pred": 0,
            "proba": 0.0,
            "severity": "",
            "messages": [HumanMessage(content=p["description"])],
            "human_approved": False,
            "final_decision": "",
        }

        result = batch_graph.invoke(initial_state, config=config)

        # Extract all tool results from messages
        policy_id = None
        severity = None
        reason = None
        recommended_fix = None
        trigger_words = None
        llm_narrative = None

        for message in reversed(result["messages"]):
            if not hasattr(message, "name") or not message.name:
                continue
            if message.name == "write_compliance_verdict" and policy_id is None:
                for line in message.content.split("\n"):
                    if line.startswith("POLICY VIOLATED"):
                        policy_id = line.split(":", 1)[1].strip()
                    elif line.startswith("SEVERITY"):
                        severity = line.split(":", 1)[1].strip()
                    elif line.startswith("REASON"):
                        reason = line.split(":", 1)[1].strip()
                    elif line.startswith("RECOMMENDED FIX"):
                        recommended_fix = line.split(":", 1)[1].strip()
            elif message.name == "get_trigger_words" and trigger_words is None:
                trigger_words = message.content
            elif message.name == "llm_assessment" and llm_narrative is None:
                llm_narrative = message.content

        timestamp = datetime.datetime.now().isoformat(timespec="seconds")

        entry = {
            "issue_key": p["issue_key"],
            "summary": p["summary"],
            "y_pred": result["y_pred"],
            "proba": result["proba"],
            "prediction": "Red Flag",
            "policy_id": policy_id,
            "severity": severity,
            "trigger_words": trigger_words,
            "reason": reason,
            "recommended_fix": recommended_fix,
            "llm_narrative": llm_narrative,
            "final_decision": result["final_decision"],
            "human_approved": result["human_approved"],
            "timestamp": timestamp,
        }
        results.append(entry)

        print(f"  Done. Policy: {policy_id}, Severity: {severity}")

        # Build and post Jira comment
        comment_lines = [
            "COMPLIANCE SENTINEL ASSESSMENT",
            "=" * 40,
            f"Issue           : {p['issue_key']}",
            f"Prediction      : Red Flag",
            f"Probability     : {result['proba']}",
            f"Policy violated : {policy_id or 'Unknown'}",
            f"Severity        : {severity or 'Unknown'}",
            "",
            "REASON:",
            reason or "See full assessment.",
            "",
            "RECOMMENDED FIX:",
            recommended_fix or "See full assessment.",
            "",
            "LLM NARRATIVE:",
            llm_narrative or "Not available.",
            "",
            "TRIGGER WORDS:",
            trigger_words or "Not available.",
            "",
            f"Decision        : {result['final_decision']}",
            f"Assessed at     : {timestamp}",
            "",
            "NOTE: This assessment was generated automatically by the Autonomous Compliance Sentinel.",
            "High severity violations are PENDING HUMAN REVIEW on the dashboard.",
        ]

        post_jira_comment(p["issue_key"], "\n".join(comment_lines))

    return results

print("Batch pipeline function defined.")

Batch pipeline function defined.


### Step 10c -- Run the batch and save results to JSON

In [25]:
# Run the full batch pipeline
all_results = run_batch_pipeline(proposals)

# Save results to a JSON file for your friend's Streamlit dashboard
output_path = r"C:\MyGithubSpace\Data-Ethics\Works\Week 4\compliance_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\nAll done. {len(all_results)} proposals processed.")
print(f"Results saved to: {output_path}")

# Summary table
print("\n--- SUMMARY ---")
print(f"{'Issue':<8} {'Prediction':<12} {'Proba':<8} {'Policy':<10} {'Severity':<10}")
print("-" * 50)
for r in all_results:
    print(f"{r['issue_key']:<8} {r['prediction']:<12} {str(r['proba']):<8} {str(r['policy_id'] or '-'):<10} {str(r['severity'] or '-'):<10}")


Processing DE2-1: Issue 1 -- RAI-02 Red Flag (Transparency)
  Signal 1: Red Flag (proba=0.5413)
  Done. Policy: RAI-01, Severity: High
  Comment posted to DE2-1 successfully.

Processing DE2-2: Issue 2 -- RAI-01 Red Flag (Data Protection)
  Signal 1: Red Flag (proba=0.4859)
  Done. Policy: RAI-01, Severity: High
  Comment posted to DE2-2 successfully.

Processing DE2-3: Issue 3 -- RAI-07 Red Flag (Human Oversight)
  Signal 1: Red Flag (proba=0.6207)


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxwsx9m4fpktxec3p3cfve1e` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5419, Requested 627. Please try again in 459.999999ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}